# Cross-seed analysis of the Fairlearn and AIF360 experiments

This notebook reads the out-of-fold predictions written by `Fairlearn.ipynb` and `AIF360.ipynb`
(one run per random seed, in `Results/seed_<k>/<library>/`) and produces everything that depends
on more than one run:

1. **Uncertainty** (Reviewer 2; Reviewer 3, comments 2-3): mean and standard deviation over the seeds,
   95% confidence intervals from a patient-level bootstrap pooled over the seeds, and a paired
   bootstrap test of every configuration against the Original Model (Holm-corrected).
2. **Library comparison** (Reviewer 2; Reviewer 3, comment 4): both libraries' metric definitions applied
   to *identical* predictions, and a matched experiment in which Fairlearn's reduction and AIF360's
   wrapper of it are run with the same estimator, constraint, sensitive features and folds.
3. **Data bias versus model bias** (Reviewer 3, comment 5): group sizes and COVID-19 base rates in the
   data, compared with the Original Model's selection and error rates.
4. **Figures** for the paper, with 95% confidence intervals, written to `Plots/`.

In [ ]:
import os, glob, json, itertools, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

BASE      = os.environ.get("MIDRC_RESULTS", os.path.join(os.getcwd(), "Results"))
PLOT_DIR  = os.environ.get("MIDRC_PLOTS", os.path.join(os.getcwd(), "Plots"))
N_BOOT    = int(os.environ.get("MIDRC_NBOOT", "500"))      # bootstrap replicates per seed
os.makedirs(PLOT_DIR, exist_ok=True)

SEEDS = sorted(int(p.split("_")[-1]) for p in glob.glob(os.path.join(BASE, "seed_*"))
               if os.path.exists(os.path.join(p, "AIF360", "Metrics_Values_AIF360.xlsx"))
               and os.path.exists(os.path.join(p, "Fairlearn", "Metrics_Values_Fairlear.csv")))
print("complete seeds:", SEEDS)

# configuration -> prediction file, in the order used by the figures
METHODS = {
    "Fairlearn": [
        ("Original Model", "y_pred_Original.csv", ""),
        ("Correlation Remover", "y_pred_Corr.csv", "Pre-processing"),
        ("Exponentiated Gradient", "y_pred_expGrad.csv", "In-processing"),
        ("Grid Search", "y_pred_gridsearch.csv", "In-processing"),
        ("Threshold Optimizer", "y_pred_Threshold.csv", "Post-processing"),
        ("Correlation Remover → Exponentiated Gradient", "y_pred_Corr_ExpGrad.csv", "Sequential Algorithms"),
        ("Grid Search → Threshold Optimizer", "y_pred_Grid_Threshold.csv", "Sequential Algorithms"),
        ("Correlation Remover → Threshold Optimizer", "y_pred_Corr_Threshold.csv", "Sequential Algorithms"),
    ],
    "AIF360": [
        ("Original", "y_pred_Original.csv", ""),
        ("Reweighing", "y_pred_Reweighing.csv", "Preprocessing Algorithms"),
        ("Disparate Impact Remover", "y_pred_DisparateImpactRemover.csv", "Preprocessing Algorithms"),
        ("Learning Fair Representations", "y_pred_LFR.csv", "Preprocessing Algorithms"),
        ("Exponentiated Gradient Reduction", "y_pred_EGR.csv", "In-processing Algorithms"),
        ("Grid Search Reduction", "y_pred_GSR.csv", "In-processing Algorithms"),
        ("Equalized Odds Postprocessing", "y_pred_EOP.csv", "Postprocessing Algorithms"),
        ("Reject Option Classification", "y_pred_ROC.csv", "Postprocessing Algorithms"),
        ("RW + EGR", "y_pred_Sequential_RW_EGR.csv", "Sequential Models"),
        ("DIR + GSR", "y_pred_Sequential_DIR_GSR.csv", "Sequential Models"),
        ("EGR + CEOP", "y_pred_Sequential_EGR_CEOP.csv", "Sequential Models"),
        ("RW + CEOP", "y_pred_Sequential_RW_CEOP.csv", "Sequential Models"),
    ],
}
GENDER = {0: "Female", 1: "Male"}
ETHNICITY = {0: "Asian", 1: "Black", 2: "Hispanic", 3: "Others", 4: "White"}
ATTRS = ["Gender", "Ethnicity", "Gender+Ethnicity"]


def labels_from(df):
    """Hard labels from a saved prediction file (score matrix -> argmax of the two
    Status columns; a single column is already a label vector)."""
    a = df.values
    if a.ndim == 2 and a.shape[1] == 1:
        return a[:, 0].astype(int)
    return np.argmax(a[:, :2], axis=1).astype(int)


def load_seed(seed):
    out = {}
    for lib, items in METHODS.items():
        d = os.path.join(BASE, f"seed_{seed}", lib)
        frame = pd.read_csv(os.path.join(d, "eval_frame.csv"))
        preds = {}
        for name, fn, _ in items:
            p = os.path.join(d, fn)
            if os.path.exists(p):
                preds[name] = labels_from(pd.read_csv(p))
        out[lib] = (frame, preds)
    return out


RUNS = {s: load_seed(s) for s in SEEDS}
frame0 = RUNS[SEEDS[0]]["Fairlearn"][0]
for s in SEEDS:                      # every run must evaluate the same images in the same order
    for lib in METHODS:
        f = RUNS[s][lib][0]
        assert (f["case_id"].values == frame0["case_id"].values).all() and (f["Status"].values == frame0["Status"].values).all()
print(len(frame0), "images |", frame0["case_id"].nunique(), "patients")

## Metric definitions

Both libraries' metrics are re-implemented here in NumPy so that they can be bootstrapped quickly.
The next cell checks every AIF360 quantity against `aif360.metrics.ClassificationMetric` on real
predictions.

* **Fairlearn** (as in `Fairlearn.ipynb`): for the COVID-19 class, the demographic parity difference
  is the largest minus the smallest selection rate across groups; the equalized odds difference is the
  larger of the TPR and FPR spreads. All groups are used.
* **AIF360** (as in `AIF360.ipynb`): one group is privileged and the union of the other listed groups is
  unprivileged; each metric is the maximum over the privileged group and the favourable class. The
  ethnicity groups are Asian, Black, Hispanic and White (*Others* is not listed, as in the notebook).
  Theil and generalized entropy (α = 2) indices use the benefit b = ŷ − y + 1 over all individuals.

In [ ]:
def group_codes(frame, attr, library):
    g, e = frame["Gender"].values, frame["Ethnicity"].values
    if attr == "Gender":
        return g, [0, 1]
    if attr == "Ethnicity":
        return e, ([0, 1, 2, 3, 4] if library == "Fairlearn" else [0, 1, 2, 4])
    code_ = g * 10 + e
    eth = [0, 1, 2, 3, 4] if library == "Fairlearn" else [0, 1, 2, 4]
    return code_, [gg * 10 + ee for gg in [0, 1] for ee in eth]


def _rates(yt, yp):
    n = len(yp)
    sr = yp.mean() if n else np.nan
    pos, neg = yt == 1, yt == 0
    tpr = yp[pos].mean() if pos.any() else np.nan
    fpr = yp[neg].mean() if neg.any() else np.nan
    return sr, tpr, fpr


def _spread(v):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    return np.nan if len(v) < 2 else v.max() - v.min()


def fairlearn_metrics(y, p, codes, groups):
    yt, yp = (y == 1).astype(int), (p == 1).astype(int)      # COVID-19 class, one-vs-rest
    r = [_rates(yt[codes == g], yp[codes == g]) for g in groups if (codes == g).any()]
    sr, tpr, fpr = map(np.array, zip(*r))
    return {"DPD": _spread(sr), "EOD": np.nanmax([_spread(tpr), _spread(fpr)])}


def _clean(v):
    return v if np.isfinite(v) else 0.0                       # same guard as the notebook


def aif360_metrics(y, p, codes, groups):
    best = dict.fromkeys(["SPD", "AOD", "EOpD", "Theil", "GEI", "DI"], -np.inf)
    listed = np.isin(codes, groups)
    for fav in (0, 1):
        yt, yp = (y == fav).astype(int), (p == fav).astype(int)
        b = yp - yt + 1.0; mu = b.mean()
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = b / mu
            theil = np.mean(np.where(ratio > 0, ratio * np.log(ratio), 0.0))
        gei = np.mean(ratio ** 2 - 1) / 2.0
        best["Theil"] = max(best["Theil"], _clean(theil)); best["GEI"] = max(best["GEI"], _clean(gei))
        for g in groups:
            priv = codes == g; unpriv = listed & ~priv
            if not priv.any() or not unpriv.any():
                continue
            srp, tprp, fprp = _rates(yt[priv], yp[priv])
            sru, tpru, fpru = _rates(yt[unpriv], yp[unpriv])
            with np.errstate(divide="ignore", invalid="ignore"):
                vals = {"SPD": sru - srp, "AOD": 0.5 * ((fpru - fprp) + (tpru - tprp)),
                        "EOpD": tpru - tprp, "DI": sru / srp}
            for k, v in vals.items():
                best[k] = max(best[k], _clean(v))
    return best


def all_metrics(frame, p, library, idx=None):
    y = frame["Status"].values
    if idx is not None:
        y, p = y[idx], p[idx]
    out = {"Accuracy": (y == p).mean()}
    for attr in ATTRS:
        codes, groups = group_codes(frame, attr, library)
        if idx is not None:
            codes = codes[idx]
        m = fairlearn_metrics(y, p, codes, groups) if library == "Fairlearn" else aif360_metrics(y, p, codes, groups)
        for k, v in m.items():
            out[f"{attr}|{k}"] = v
    return out

In [ ]:
# ── Check the NumPy AIF360 metrics against aif360 itself (seed 0, Original model) ──
from aif360.datasets import StandardDataset
from aif360.metrics import ClassificationMetric

f, preds = RUNS[SEEDS[0]]["AIF360"]
p = preds["Original"]
df = f[["Gender", "Ethnicity"]].copy(); df["Status"] = f["Status"].values
codes, groups = group_codes(f, "Ethnicity", "AIF360")
ref = dict.fromkeys(["SPD", "AOD", "EOpD", "Theil", "GEI", "DI"], -np.inf)
for fav in (0, 1):
    for gp in groups:
        ds = StandardDataset(df, label_name="Status", favorable_classes=[fav],
                             protected_attribute_names=["Ethnicity"], privileged_classes=[[gp]])
        pds = ds.copy(); pds.labels = p.astype(float).reshape(-1, 1)   # raw labels; aif360 compares with favorable_label
        gl = [{"Ethnicity": float(g)} for g in groups]
        i = groups.index(gp)
        m = ClassificationMetric(ds, pds, unprivileged_groups=gl[:i] + gl[i + 1:], privileged_groups=[gl[i]])
        for k, fn in [("SPD", m.statistical_parity_difference), ("AOD", m.average_odds_difference),
                      ("EOpD", m.equal_opportunity_difference), ("Theil", m.theil_index),
                      ("GEI", m.generalized_entropy_index), ("DI", m.disparate_impact)]:
            v = fn(); ref[k] = max(ref[k], v if np.isfinite(v) else 0.0)
mine = aif360_metrics(f["Status"].values, p, codes, groups)
check = pd.DataFrame({"aif360": ref, "numpy": mine}); check["abs diff"] = (check.aif360 - check.numpy).abs()
print(check.round(6)); assert check["abs diff"].max() < 1e-9, "metric re-implementation does not match aif360"

## 1. Point estimates, seed variability and bootstrap confidence intervals

In [ ]:
# patient-level bootstrap: resample patients (all images of a resampled patient come along)
case = frame0["case_id"].values
uniq, inv = np.unique(case, return_inverse=True)
rows_of = [np.flatnonzero(inv == k) for k in range(len(uniq))]

def boot_index(rng):
    pick = rng.integers(0, len(uniq), len(uniq))
    return np.concatenate([rows_of[k] for k in pick])

records, boot = [], []          # point values per seed / bootstrap values per seed x replicate
for s in SEEDS:
    rng = np.random.default_rng(1000 + s)
    idxs = [boot_index(rng) for _ in range(N_BOOT)]
    for lib, (frame, preds) in RUNS[s].items():
        for name, p in preds.items():
            records.append({"seed": s, "library": lib, "method": name, **all_metrics(frame, p, lib)})
            for b, idx in enumerate(idxs):
                boot.append({"seed": s, "b": b, "library": lib, "method": name, **all_metrics(frame, p, lib, idx)})
    print("seed", s, "done", flush=True)

point = pd.DataFrame(records)
bootdf = pd.DataFrame(boot)
point.to_csv(os.path.join(BASE, "point_estimates_by_seed.csv"), index=False)
print(point.groupby(["library", "method"], sort=False)["Accuracy"].agg(["mean", "std"]).round(4))

In [ ]:
# Summary: mean and SD over seeds, pooled bootstrap 95% CI, paired bootstrap test vs Original
def holm(pvals):
    """Holm-Bonferroni adjusted p-values."""
    p = np.asarray(pvals, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

metric_cols = [c for c in point.columns if c == "Accuracy" or "|" in c]
ORIGINAL = {"Fairlearn": "Original Model", "AIF360": "Original"}
rows = []
for lib, items in METHODS.items():
    for name, _, stage in items:
        P = point[(point.library == lib) & (point.method == name)]
        B = bootdf[(bootdf.library == lib) & (bootdf.method == name)].sort_values(["seed", "b"])
        B0 = bootdf[(bootdf.library == lib) & (bootdf.method == ORIGINAL[lib])].sort_values(["seed", "b"])
        if P.empty:
            continue
        for col in metric_cols:
            bv, b0 = B[col].values, B0[col].values
            diff = bv - b0
            p = np.nan if name == ORIGINAL[lib] else min(1.0, 2 * min((diff <= 0).mean(), (diff >= 0).mean()))
            rows.append({"library": lib, "method": name, "stage": stage, "metric": col,
                         "mean": P[col].mean(), "sd_seeds": P[col].std(ddof=1) if len(P) > 1 else np.nan,
                         "ci_low": np.nanpercentile(bv, 2.5), "ci_high": np.nanpercentile(bv, 97.5),
                         "diff_vs_original": P[col].mean() - point[(point.library == lib) & (point.method == ORIGINAL[lib])][col].mean(),
                         "diff_ci_low": np.nanpercentile(diff, 2.5), "diff_ci_high": np.nanpercentile(diff, 97.5),
                         "p_vs_original": p})
summary = pd.DataFrame(rows)
# Holm correction within each library x metric family (all configurations compared with the original)
summary["p_holm"] = np.nan
for (lib, met), g in summary.groupby(["library", "metric"]):
    ok = g["p_vs_original"].notna()
    if ok.any():
        summary.loc[g.index[ok], "p_holm"] = holm(g.loc[ok, "p_vs_original"])
summary.to_csv(os.path.join(BASE, "summary_with_CI.csv"), index=False)
with pd.ExcelWriter(os.path.join(BASE, "summary_with_CI.xlsx")) as xw:
    for lib in METHODS:
        summary[summary.library == lib].to_excel(xw, sheet_name=lib, index=False)
summary[summary.metric == "Accuracy"][["library", "method", "mean", "sd_seeds", "ci_low", "ci_high", "p_holm"]].round(4)

In [ ]:
# Discrimination of the Original Model (scores are available for it): AUC per seed
for s in SEEDS:
    d = os.path.join(BASE, f"seed_{s}", "Fairlearn")
    sc = pd.read_csv(os.path.join(d, "y_pred_Original.csv")).values[:, 1]
    print(f"seed {s}: AUC = {roc_auc_score(frame0['Status'].values, sc):.4f}")

## 2. Why do the two libraries disagree?

### 2a. Same predictions, both metric definitions

In [ ]:
# For each pair of corresponding configurations, apply BOTH libraries' metric definitions
# to the SAME predictions. Differences within a row come from the metric definition;
# differences between the two rows of a pair come from the algorithm/implementation.
PAIRS = [("Original Model", "Original"), ("Exponentiated Gradient", "Exponentiated Gradient Reduction"),
         ("Grid Search", "Grid Search Reduction")]
rows = []
for s in SEEDS:
    for fl_name, aif_name in PAIRS:
        for lib, name in [("Fairlearn", fl_name), ("AIF360", aif_name)]:
            frame, preds = RUNS[s][lib]
            if name not in preds:
                continue
            p = preds[name]
            for attr in ATTRS:
                cf, gf = group_codes(frame, attr, "Fairlearn"); ca, ga = group_codes(frame, attr, "AIF360")
                fm = fairlearn_metrics(frame["Status"].values, p, cf, gf)
                am = aif360_metrics(frame["Status"].values, p, ca, ga)
                rows.append({"seed": s, "pair": f"{fl_name} / {aif_name}", "predictions from": f"{lib}: {name}",
                             "attribute": attr, "Accuracy": (frame["Status"].values == p).mean(),
                             "Fairlearn DPD": fm["DPD"], "Fairlearn EOD": fm["EOD"],
                             "AIF360 SPD": am["SPD"], "AIF360 AOD": am["AOD"], "AIF360 EOpD": am["EOpD"]})
same_pred = (pd.DataFrame(rows).groupby(["pair", "predictions from", "attribute"], sort=False)
             .mean(numeric_only=True).drop(columns="seed"))
same_pred.to_csv(os.path.join(BASE, "library_same_predictions.csv"))
same_pred.round(4)

### 2b. Matched experiment: same reduction, same settings, both libraries

AIF360's `ExponentiatedGradientReduction` and `GridSearchReduction` call Fairlearn's reductions
internally. In the main notebooks the two libraries were run as each is typically configured, and
those configurations differ (below). Here both are run with an identical, fast estimator (logistic
regression), the same constraint (equalized odds), the same sensitive features (gender and ethnicity
jointly), the same budget and the same patient-grouped folds, so any remaining difference is due to
the implementation alone.

In [ ]:
import random as _random
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold
from fairlearn.reductions import ExponentiatedGradient, GridSearch, EqualizedOdds
from aif360.algorithms.inprocessing import ExponentiatedGradientReduction, GridSearchReduction

# configuration differences in the main notebooks
config = pd.DataFrame([
    ["Sensitive features used by the reductions", "Gender and Ethnicity (joint)", "Gender only"],
    ["Grid Search size", "40", "5 (GSR_GRID_SIZE)"],
    ["Exponentiated Gradient iterations", "10", "10 (EGR_MAX_ITER)"],
    ["Binary task handling", "one model per Status class (one-vs-rest)", "one model per Status class (one-vs-rest)"],
    ["Fairness metric groups (ethnicity)", "all 5 groups, max - min", "Asian, Black, Hispanic, White; one vs rest"],
    ["Fairness metric aggregation", "spread across groups", "max over privileged group and favourable class"],
], columns=["Setting", "Fairlearn notebook", "AIF360 notebook"])
display(config)

# rebuild the feature matrix exactly as the notebooks do (seed 0)
meta = pd.read_csv(os.environ.get("MIDRC_DATA_CSV", r"C:\midrc\dataset_final.csv"))
feat = np.load(os.environ.get("MIDRC_FEATURES", r"C:\midrc\features_xrv_densenet121.npz"), allow_pickle=True)
emb = pd.DataFrame(feat["embeddings"], index=feat["fname"])
meta = meta[meta["fname"].isin(emb.index)].reset_index(drop=True)
if os.environ.get("MIDRC_SMOKE"):
    meta = meta.sample(int(os.environ["MIDRC_SMOKE"]), random_state=42).reset_index(drop=True)
assert (meta["case_id"].values == frame0["case_id"].values).all()
P_COMP = int(os.environ.get("MIDRC_PCA", "16"))
Xm = frame0[["Age", "Gender", "Ethnicity"]].copy()
Xm[[f"PCA{i}" for i in range(P_COMP)]] = PCA(n_components=P_COMP, random_state=0).fit_transform(
    emb.loc[meta["fname"]].values)
ym = frame0["Status"].values
Gm = frame0["case_id"].values
SF = Xm["Gender"].astype(str) + "|" + Xm["Ethnicity"].astype(str)

def lr():
    return LogisticRegression(max_iter=2000)

out = {k: np.zeros(len(ym), int) for k in ["FL ExpGrad", "AIF EGR", "FL GridSearch", "AIF GSR"]}
for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=0).split(Xm, ym, Gm):
    Xtr, Xte = Xm.iloc[tr].reset_index(drop=True), Xm.iloc[te].reset_index(drop=True)
    ytr = pd.Series(ym[tr], name="Status")
    # Fairlearn
    np.random.seed(0); _random.seed(0)
    eg = ExponentiatedGradient(lr(), constraints=EqualizedOdds(), eps=0.01, max_iter=50)
    eg.fit(Xtr, ytr, sensitive_features=SF.iloc[tr].values)
    out["FL ExpGrad"][te] = eg.predict(Xte, random_state=0)
    gs = GridSearch(lr(), constraints=EqualizedOdds(), grid_size=20)
    gs.fit(Xtr, ytr, sensitive_features=SF.iloc[tr].values)
    out["FL GridSearch"][te] = gs.predict(Xte)
    # AIF360 (dataset API, same estimator / constraint / budget; both attributes protected)
    def ds(Xp, yp):
        return StandardDataset(pd.concat([Xp, pd.Series(yp, name="Status")], axis=1), label_name="Status",
                               favorable_classes=[1], protected_attribute_names=["Gender", "Ethnicity"],
                               privileged_classes=[[1], [4]])
    np.random.seed(0); _random.seed(0)
    egr = ExponentiatedGradientReduction(estimator=lr(), constraints="EqualizedOdds", eps=0.01, max_iter=50,
                                         drop_prot_attr=False)
    egr.fit(ds(Xtr, ytr.values))
    out["AIF EGR"][te] = egr.predict(ds(Xte, ym[te])).labels.ravel().astype(int)
    gsr = GridSearchReduction(estimator=lr(), constraints="EqualizedOdds", grid_size=20, drop_prot_attr=False)
    gsr.fit(ds(Xtr, ytr.values))
    out["AIF GSR"][te] = gsr.predict(ds(Xte, ym[te])).labels.ravel().astype(int)

rows = []
for a, b in [("FL ExpGrad", "AIF EGR"), ("FL GridSearch", "AIF GSR")]:
    for name in (a, b):
        r = {"run": name, "agreement with partner": (out[a] == out[b]).mean()}
        r.update({k: v for k, v in all_metrics(frame0, out[name], "Fairlearn").items()})
        rows.append(r)
matched = pd.DataFrame(rows).set_index("run")
matched.to_csv(os.path.join(BASE, "library_matched_experiment.csv"))
matched.round(4)

## 3. Data bias versus model bias

In [ ]:
f, preds = RUNS[SEEDS[0]]["Fairlearn"]
y = f["Status"].values
rows = []
for attr in ["Gender", "Ethnicity"]:
    codes = f[attr].values; names = GENDER if attr == "Gender" else ETHNICITY
    for g in sorted(np.unique(codes)):
        m = codes == g
        sel = np.mean([RUNS[s]["Fairlearn"][1]["Original Model"][m].mean() for s in SEEDS])
        tpr = np.mean([RUNS[s]["Fairlearn"][1]["Original Model"][m & (y == 1)].mean() for s in SEEDS])
        fpr = np.mean([RUNS[s]["Fairlearn"][1]["Original Model"][m & (y == 0)].mean() for s in SEEDS])
        rows.append({"attribute": attr, "group": names[g], "n": m.sum(), "share": m.mean(),
                     "COVID-19 base rate (data)": y[m].mean(), "predicted COVID-19 rate (model)": sel,
                     "TPR": tpr, "FPR": fpr})
groups_tbl = pd.DataFrame(rows)
groups_tbl.to_csv(os.path.join(BASE, "data_vs_model_by_group.csv"), index=False)

summ = []
for attr in ["Gender", "Ethnicity"]:
    t = groups_tbl[groups_tbl.attribute == attr]
    data_gap = t["COVID-19 base rate (data)"].max() - t["COVID-19 base rate (data)"].min()
    model_gap = t["predicted COVID-19 rate (model)"].max() - t["predicted COVID-19 rate (model)"].min()
    summ.append({"attribute": attr, "base-rate gap in the data": data_gap,
                 "selection-rate gap of the model (DPD)": model_gap,
                 "model gap / data gap": model_gap / data_gap if data_gap else np.nan,
                 "corr(base rate, predicted rate) across groups":
                     np.corrcoef(t["COVID-19 base rate (data)"], t["predicted COVID-19 rate (model)"])[0, 1]})
data_bias = pd.DataFrame(summ)
data_bias.to_csv(os.path.join(BASE, "data_vs_model_summary.csv"), index=False)
display(groups_tbl.round(3)); data_bias.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [2, 5]})
for ax, attr in zip(axes, ["Gender", "Ethnicity"]):
    t = groups_tbl[groups_tbl.attribute == attr]
    x = np.arange(len(t)); w = 0.38
    ax.bar(x - w / 2, 100 * t["COVID-19 base rate (data)"], w, color="#7b9cda", edgecolor="black", hatch="///", label="COVID-19 base rate in the data")
    ax.bar(x + w / 2, 100 * t["predicted COVID-19 rate (model)"], w, color="#f4845f", edgecolor="black", hatch="xxx", label="Predicted COVID-19 rate (Original Model)")
    for xi, n in zip(x, t["n"]):
        ax.text(xi, 2, f"n={n}", ha="center", fontsize=8, color="white", fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(t["group"]); ax.set_ylim(0, 100); ax.set_title(attr)
    ax.set_ylabel("Rate (%)"); ax.grid(axis="y", linestyle="--", alpha=0.4)
axes[0].legend(loc="upper left", fontsize=8)
fig.suptitle("Data bias versus model bias", fontsize=13)
fig.tight_layout(); fig.savefig(os.path.join(PLOT_DIR, "Data-vs-Model-Bias.png"), dpi=150, bbox_inches="tight"); plt.show()

## 4. Figures (mean over seeds, 95% bootstrap CI)

In [ ]:
COLORS = ["#2ec4b6", "#f4845f", "#7b9cda", "#d966a0", "#8bc34a"]
HATCHES = ["///", "///", "xxx", "...", "///"]
import textwrap

def get(lib, name, col):
    r = summary[(summary.library == lib) & (summary.method == name) & (summary.metric == col)]
    return (r["mean"].iloc[0], r["ci_low"].iloc[0], r["ci_high"].iloc[0], r["p_holm"].iloc[0]) if len(r) else (np.nan,) * 4

def plot(lib, cols, labels, title, fname, ylabel="Metric value (%)"):
    items = [(n, st) for n, _, st in METHODS[lib] if not summary[(summary.library == lib) & (summary.method == n)].empty]
    names = [n for n, _ in items]; cats = [st for _, st in items]
    n, k = len(names), len(cols)
    bw = min(0.8 / k, 0.22); x = np.arange(n); off = (np.arange(k) - (k - 1) / 2) * bw
    fig, ax1 = plt.subplots(figsize=(max(12, 1.7 * n), 7.5)); ax2 = ax1.twinx()
    top = 0
    for j, (col, lab) in enumerate(zip(cols, labels)):
        v = np.array([get(lib, nm, col) for nm in names], float)
        m, lo, hi, p = v[:, 0] * 100, v[:, 1] * 100, v[:, 2] * 100, v[:, 3]
        ax1.bar(x + off[j], m, bw, color=COLORS[j % 5], hatch=HATCHES[j % 5], edgecolor="black", linewidth=0.6, label=lab, zorder=3)
        ax1.errorbar(x + off[j], m, yerr=[m - lo, hi - m], fmt="none", ecolor="black", elinewidth=0.9, capsize=2.5, zorder=4)
        for xi, mi, hii, pi in zip(x + off[j], m, hi, p):
            star = "*" if (np.isfinite(pi) and pi < 0.05) else ""
            ax1.text(xi, hii + 0.6, f"{mi:.1f}{star}", ha="center", va="bottom", fontsize=6.5, rotation=90)
        top = max(top, np.nanmax(hi))
    acc = np.array([get(lib, nm, "Accuracy") for nm in names], float) * 100
    ax2.errorbar(x, acc[:, 0], yerr=[acc[:, 0] - acc[:, 1], acc[:, 2] - acc[:, 0]], color="black", marker="o", ms=6, lw=1.6, capsize=3, zorder=5)
    for xi, a in zip(x, acc[:, 0]):
        ax2.text(xi, a + 2, f"{a:.1f}%", ha="center", fontsize=8.5, fontweight="bold")
    ax1.set_ylim(0, max(10, np.ceil(top * 1.35 / 5) * 5)); ax2.set_ylim(0, 100)
    ax1.set_xlim(-0.5, n - 0.5); ax1.set_xticks(x); ax1.set_xticklabels([textwrap.fill(s, 16) for s in names], fontsize=8.5)
    ax1.set_ylabel(ylabel); ax2.set_ylabel("Accuracy (%)"); ax1.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
    start = 0
    for i in range(1, n + 1):
        if i == n or cats[i] != cats[start]:
            if cats[start]:
                ax1.text((start + i - 1) / 2, -0.17, cats[start], transform=ax1.get_xaxis_transform(), ha="center", va="top", fontsize=9, style="italic", color="#444")
            if i < n:
                ax1.axvline(i - 0.5, color="#999", ls=":", lw=1)
            start = i
    handles = [mpatches.Patch(facecolor=COLORS[j % 5], hatch=HATCHES[j % 5], edgecolor="black", label=l) for j, l in enumerate(labels)]
    handles.append(plt.Line2D([0], [0], color="black", marker="o", label="Accuracy"))
    ax1.legend(handles=handles, loc="upper left", fontsize=8.5, framealpha=0.9)
    ax1.set_title(title + f"  (mean of {len(SEEDS)} seeds, 95% CI; * p<0.05 vs. original, Holm)", fontsize=12)
    fig.tight_layout(); fig.savefig(os.path.join(PLOT_DIR, fname), dpi=150, bbox_inches="tight"); plt.show()

for attr, tag in [("Gender", "Gender"), ("Ethnicity", "Ethnicity"), ("Gender+Ethnicity", "Gender-Ethnicity")]:
    plot("Fairlearn", [f"{attr}|DPD"], ["Demographic Parity Difference (COVID-19)"], f"Fairlearn - {attr}",
         f"Fairlearn-{tag}-Demographic_Parity.png", "Demographic Parity (%)")
    plot("Fairlearn", [f"{attr}|EOD"], ["Equalized Odds Difference (COVID-19)"], f"Fairlearn - {attr}",
         f"Fairlearn-{tag}-Equalized_Odds.png", "Equalized Odds (%)")
    plot("AIF360", [f"{attr}|{m}" for m in ["SPD", "AOD", "EOpD", "Theil", "GEI"]],
         ["Statistical Parity Difference", "Average Odds Difference", "Equal Opportunity Difference",
          "Theil Index", "Generalized Entropy Index"], f"AIF360 - {attr}", f"AIF360-{tag}.png", "Metrics Values (%)")
print(sorted(os.listdir(PLOT_DIR)))